In [1]:
import pandas as pd
import pyarrow as pa
import numpy

In [2]:
df = pd.read_csv("../data/raw/cinema_dataset.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 142524 entries, 0 to 142523
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   film_code     142524 non-null  int64  
 1   cinema_code   142524 non-null  int64  
 2   total_sales   142524 non-null  int64  
 3   tickets_sold  142524 non-null  int64  
 4   tickets_out   142524 non-null  int64  
 5   show_time     142524 non-null  int64  
 6   occu_perc     142399 non-null  float64
 7   ticket_price  142524 non-null  float64
 8   ticket_use    142524 non-null  int64  
 9   capacity      142399 non-null  float64
 10  date          142524 non-null  str    
 11  month         142524 non-null  int64  
 12  quarter       142524 non-null  int64  
 13  day           142524 non-null  int64  
dtypes: float64(3), int64(10), str(1)
memory usage: 16.6 MB


In [4]:
df.head()

,film_code,cinema_code,total_sales,tickets_sold,tickets_out,show_time,occu_perc,ticket_price,ticket_use,capacity,date,month,quarter,day
0,1492,304,3900000,26,0,4,4.26,150000.0,26,610.328638,2018-05-05,5,2,5
1,1492,352,3360000,42,0,5,8.08,80000.0,42,519.801980,2018-05-05,5,2,5
2,1492,489,2560000,32,0,4,20.00,80000.0,32,160.000000,2018-05-05,5,2,5
3,1492,429,1200000,12,0,1,11.01,100000.0,12,108.991826,2018-05-05,5,2,5
4,1492,524,1200000,15,0,3,16.67,80000.0,15,89.982004,2018-05-05,5,2,5


In [5]:
df_clean = df.copy()

In [6]:
df_clean["date"] = pd.to_datetime(df_clean["date"])

In [7]:
df_clean = df_clean.sort_values("date")

In [8]:
df_clean.isnull().sum()

film_code         0
cinema_code       0
total_sales       0
tickets_sold      0
tickets_out       0
show_time         0
occu_perc       125
ticket_price      0
ticket_use        0
capacity        125
date              0
month             0
quarter           0
day               0
dtype: int64

In [9]:
df_clean = df_clean.dropna()

In [10]:
# verficiar que la variable occu_per sea correcta
df_clean["occu_cal"] = (df_clean["tickets_sold"]/df_clean["capacity"]) * 100
df_clean[["occu_cal", "occu_perc"]].head()

,occu_cal,occu_perc
46143,47.76,47.76
46142,2.07,2.07
104016,21.42,21.42
72077,0.21,0.21
72076,0.73,0.73


In [11]:
df_clean.head()


,film_code,cinema_code,total_sales,tickets_sold,tickets_out,show_time,occu_perc,ticket_price,ticket_use,capacity,date,month,quarter,day,occu_cal
46143,1471,448,32030000,267,0,2,47.76,119962.546816,267,559.045226,2018-02-21,2,1,21,47.76
46142,1471,518,180000,3,0,1,2.07,60000.000000,3,144.927536,2018-02-23,2,1,23,2.07
104016,1485,304,43200000,363,0,8,21.42,119008.264463,363,1694.677871,2018-03-14,3,1,14,21.42
72077,1483,83,300000,3,0,10,0.21,100000.000000,3,1428.571429,2018-03-14,3,1,14,0.21
72076,1483,262,560000,7,0,3,0.73,80000.000000,7,958.904110,2018-03-14,3,1,14,0.73


In [12]:
# Validación para ver si el ticket coincide con total ventas 
df_clean["excepted_sales"] = df_clean["ticket_price"] * df_clean["tickets_sold"]
df_clean["error"] = df_clean["total_sales"] - df_clean["excepted_sales"]
df_clean["error"].describe()

count    1.423990e+05
mean    -2.247183e-12
std      2.136895e-09
min     -1.192093e-07
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.192093e-07
Name: error, dtype: float64

In [13]:
# Escalar el precio del ticket y total ventas a un valor adecuado para el analisis (desconocemos que moneda este usando para dar ese valor)
df_clean["ticket_price"] = df_clean["ticket_price"] / 1000
df_clean["total_sales"] = df_clean["total_sales"] / 1000
df_clean.head()

,film_code,cinema_code,total_sales,tickets_sold,tickets_out,show_time,occu_perc,ticket_price,ticket_use,capacity,date,month,quarter,day,occu_cal,excepted_sales,error
46143,1471,448,32030.0,267,0,2,47.76,119.962547,267,559.045226,2018-02-21,2,1,21,47.76,32030000.0,0.0
46142,1471,518,180.0,3,0,1,2.07,60.000000,3,144.927536,2018-02-23,2,1,23,2.07,180000.0,0.0
104016,1485,304,43200.0,363,0,8,21.42,119.008264,363,1694.677871,2018-03-14,3,1,14,21.42,43200000.0,0.0
72077,1483,83,300.0,3,0,10,0.21,100.000000,3,1428.571429,2018-03-14,3,1,14,0.21,300000.0,0.0
72076,1483,262,560.0,7,0,3,0.73,80.000000,7,958.904110,2018-03-14,3,1,14,0.73,560000.0,0.0


In [14]:
# Guardar el df para el analisis de datos
df_final = df_clean[
    [
        'film_code',
        'cinema_code',
        'date',
        'month',
        'day',
        'show_time',
        'capacity',
        'tickets_out',
        'tickets_sold',
        'ticket_use',
        'ticket_price',
        'total_sales',
        'occu_perc'
    ]
]

In [15]:
# Guardar lo que se uso para auditar la calidad de los datos
df_audit = df_clean[[
    'film_code',
    'cinema_code',
    'date',
    'occu_perc',
    'occu_cal',
    'excepted_sales',
    'error'
]]

In [16]:
df_final.to_parquet("../data/processed/cinema_dataset_processed.parquet", index=False)
df_audit.to_parquet("../data/processed/cinema_dataset_audit.parquet", index=False)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 142524 entries, 0 to 142523
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   film_code     142524 non-null  int64  
 1   cinema_code   142524 non-null  int64  
 2   total_sales   142524 non-null  int64  
 3   tickets_sold  142524 non-null  int64  
 4   tickets_out   142524 non-null  int64  
 5   show_time     142524 non-null  int64  
 6   occu_perc     142399 non-null  float64
 7   ticket_price  142524 non-null  float64
 8   ticket_use    142524 non-null  int64  
 9   capacity      142399 non-null  float64
 10  date          142524 non-null  str    
 11  month         142524 non-null  int64  
 12  quarter       142524 non-null  int64  
 13  day           142524 non-null  int64  
dtypes: float64(3), int64(10), str(1)
memory usage: 16.6 MB
